# CertCF Random Anchor k-Sweep Analysis

Visual analysis for the random-anchor CertCF benchmark:

- input-space random anchor selection
- `k_per_class ∈ {100, 200, 500, 1000, 2000}`
- datasets: `adult`, `compas`, `german_credit`

The goal is to understand whether increasing the random atlas support improves L1/L2 proximity, and where the runtime cost is paid.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebooks.utils import load_result, plot_proximity_kdes_by_dataset, prepare_benchmark_df, setup_notebook_style

setup_notebook_style()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

RESULT_NAME = "benchmark_meeting_certcf_random_k_sweep_adult.parquet"
OLD_KMEDOIDS_NAME = "benchmark_meeting_certcf_input_vs_latent_200q.parquet"

RESULT_PATH = PROJECT_ROOT / "results" / RESULT_NAME
OLD_KMEDOIDS_PATH = PROJECT_ROOT / "results" / OLD_KMEDOIDS_NAME

SAVE_FIGS = False
FIG_DIR = PROJECT_ROOT / "results" / "figures" / "random_k_sweep"
if SAVE_FIGS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ORDER = ["adult", "compas", "german_credit"]
K_ORDER = [100, 200, 500, 1000, 2000]
K_CMAP = plt.get_cmap("plasma")
K_COLORS = {
    k: K_CMAP(i / (len(K_ORDER) - 1))
    for i, k in enumerate(K_ORDER)
}

def savefig(name: str):
    if SAVE_FIGS:
        plt.savefig(FIG_DIR / name, dpi=180, bbox_inches="tight")


## Load Results

In [ ]:
df = load_result(RESULT_PATH)

if "dataset" not in df.columns:
    raise ValueError("The combined benchmark parquet must contain a 'dataset' column.")

run_name = df["run_name"].fillna(df["method"].astype(str))
k_values = run_name.str.extract(r"k_per_class=(\d+)")[0].astype(int)
df["k_per_class_value"] = k_values
df["k_per_class"] = pd.Categorical(k_values, categories=K_ORDER, ordered=True)
df = prepare_benchmark_df(df, dataset_order=DATASET_ORDER)
success_df = prepare_benchmark_df(df, dataset_order=DATASET_ORDER, success_only_rows=True)

print(f"Loaded {len(df):,} rows from {RESULT_PATH}")
display(
    df.groupby(["dataset", "k_per_class"], observed=True)["success"]
    .agg(valid="sum", total="count", validity_pct=lambda s: 100 * s.mean())
    .reset_index()
)


## Summary Tables

In [ ]:
def q(p):
    return lambda x: x.quantile(p)

proximity_summary = (
    success_df.groupby(["dataset", "k_per_class"], observed=True)
    .agg(
        n=("success", "size"),
        l1_mean=("l1_distance", "mean"),
        l1_median=("l1_distance", "median"),
        l1_p90=("l1_distance", q(0.90)),
        l1_p95=("l1_distance", q(0.95)),
        l2_mean=("l2_distance", "mean"),
        l2_median=("l2_distance", "median"),
        l2_p90=("l2_distance", q(0.90)),
        l2_p95=("l2_distance", q(0.95)),
        l0_mean=("l0_sparsity", "mean"),
    )
    .reset_index()
)

display(proximity_summary.style.format(precision=4))

In [ ]:
time_summary = (
    success_df.groupby(["dataset", "k_per_class"], observed=True)
    .agg(
        build_mean_s=("build_time_s", "mean"),
        runtime_mean_s=("runtime_s", "mean"),
        runtime_median_s=("runtime_s", "median"),
        runtime_p90_s=("runtime_s", q(0.90)),
        projection_ms_mean=("meta__projection_time_ms", "mean"),
        search_ms_mean=("meta__search_time_ms", "mean"),
        candidates_total_mean=("meta__n_candidates_total", "mean"),
        candidates_considered_mean=("meta__n_candidates_considered", "mean"),
        qp_solved_mean=("meta__n_qp_solved", "mean"),
    )
    .reset_index()
)

display(time_summary.style.format(precision=4))

## Normalized L1/L2 Mean Trend vs k

Raw L1/L2 scales are dataset-specific, so plotting all datasets on the same absolute axis is misleading. Here each dataset is normalized by its own `k_per_class=100` value. Lower is better, and every curve starts at 1.0.

In [ ]:
normalized_proximity = proximity_summary.copy()
normalized_proximity["k_per_class_value"] = normalized_proximity["k_per_class"].astype(int)

for metric in ["l1_mean", "l2_mean"]:
    baseline = (
        normalized_proximity.loc[normalized_proximity["k_per_class_value"] == K_ORDER[0]]
        .set_index("dataset")[metric]
    )
    normalized_proximity[f"{metric}_relative"] = normalized_proximity.apply(
        lambda row: row[metric] / baseline.loc[row["dataset"]], axis=1
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)

for dataset, ds_summary in normalized_proximity.groupby("dataset", observed=True):
    ds_summary = ds_summary.sort_values("k_per_class_value")
    k = ds_summary["k_per_class_value"].to_numpy(dtype=float)
    axes[0].plot(k, ds_summary["l1_mean_relative"], marker="o", linewidth=2.2, label=str(dataset))
    axes[1].plot(k, ds_summary["l2_mean_relative"], marker="o", linewidth=2.2, label=str(dataset))

axes[0].set_title("Relative mean L1 proximity vs k")
axes[1].set_title("Relative mean L2 proximity vs k")

for ax, ylabel in zip(axes, ["Relative mean L1 (k=100 = 1.0)", "Relative mean L2 (k=100 = 1.0)"]):
    ax.axhline(1.0, color="0.45", linestyle="--", linewidth=1.2, alpha=0.8)
    ax.set_xscale("log", base=10)
    ax.set_xlim(min(K_ORDER) * 0.9, max(K_ORDER) * 1.1)
    ax.set_xticks(K_ORDER)
    ax.set_xticklabels([str(k) for k in K_ORDER])
    ax.minorticks_off()
    ax.set_xlabel("k_per_class")
    ax.set_ylabel(ylabel)
    ax.legend(title="dataset")

fig.suptitle("Random CertCF anchors: relative proximity vs k", y=1.03, fontsize=14)
# plt.tight_layout()
# savefig("normalized_mean_proximity_vs_k.png")
plt.show()

## KDE Distribution Of Proximity

Means can be misleading, so this cell shows smooth KDE estimates of the L1/L2 proximity distributions for each dataset and each `k_per_class`. The KDE makes distribution shifts easier to read than overlapping histograms.

In [ ]:
kde_df = success_df.copy()
kde_df["k_label"] = kde_df["k_per_class"].astype(int).map(lambda k: f"k={k}")
k_label_order = [f"k={k}" for k in K_ORDER]
k_palette = {f"k={k}": K_COLORS[k] for k in K_ORDER}

figures = plot_proximity_kdes_by_dataset(
    kde_df,
    dataset_order=DATASET_ORDER,
    method_order=k_label_order,
    palette=k_palette,
    method_col="k_label",
    method_labels={label: label for label in k_label_order},
)
for _, fig, _ in figures:
    savefig(f"{fig._suptitle.get_text().split(':')[0] if fig._suptitle else 'kde'}.png")
    plt.show()


## Boxplots: Median And Tail Movement

The histograms show shape. These boxplots make it easier to see whether the whole distribution moves or only a few outliers change.

In [ ]:
for metric, label in [("l1_distance", "L1 proximity"), ("l2_distance", "L2 proximity")]:
    fig, axes = plt.subplots(1, len(DATASET_ORDER), figsize=(15, 4.5), sharey=False)
    for ax, dataset in zip(axes, DATASET_ORDER):
        ds = success_df.loc[success_df["dataset"] == dataset]
        groups = [ds.loc[ds["k_per_class"].astype(int) == k, metric].to_numpy() for k in K_ORDER]
        ax.boxplot(groups, tick_labels=[str(k) for k in K_ORDER], showfliers=False)
        ax.set_title(dataset)
        ax.set_xlabel("k_per_class")
        ax.set_ylabel(label)
    fig.suptitle(f"{label} distribution by k", y=1.03, fontsize=14)
    plt.tight_layout()
    savefig(f"boxplot_{metric}_by_k.png")
    plt.show()

## Query And Build Time

This separates the two costs:

- build time: atlas construction, mostly LiRPA/polytope construction after random selection
- query time: candidate pruning plus convex projection/QP solving

In [ ]:
time_plot_summary = time_summary.copy()
time_plot_summary["k_per_class_value"] = time_plot_summary["k_per_class"].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)

for dataset, ds_summary in time_plot_summary.groupby("dataset", observed=True):
    ds_summary = ds_summary.sort_values("k_per_class_value")
    k = ds_summary["k_per_class_value"].to_numpy(dtype=float)
    axes[0].plot(k, ds_summary["build_mean_s"], marker="o", linewidth=2.2, label=str(dataset))
    axes[1].plot(k, ds_summary["runtime_mean_s"], marker="o", linewidth=2.2, label=str(dataset))

for ax, ylabel in zip(axes, ["Mean build time (s)", "Mean query time (s)"]):
    ax.set_xscale("log", base=10)
    ax.set_xlim(min(K_ORDER) * 0.9, max(K_ORDER) * 1.1)
    ax.set_xticks(K_ORDER)
    ax.set_xticklabels([str(k) for k in K_ORDER])
    ax.minorticks_off()
    ax.set_xlabel("k_per_class")
    ax.set_ylabel(ylabel)
    ax.legend(title="dataset")

fig.suptitle("Random CertCF anchors: build/query time vs k", y=1.03, fontsize=14)
plt.tight_layout()
savefig("time_vs_k.png")
plt.show()

## Where Query Time Goes

The search phase should be tiny. The projection/QP phase is expected to dominate.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True)

for dataset, ds_summary in time_summary.groupby("dataset", observed=True):
    k = ds_summary["k_per_class"].astype(int)
    axes[0].plot(k, ds_summary["search_ms_mean"], marker="o", linewidth=2.2, label=str(dataset))
    axes[1].plot(k, ds_summary["projection_ms_mean"], marker="o", linewidth=2.2, label=str(dataset))
    axes[2].plot(k, ds_summary["qp_solved_mean"], marker="o", linewidth=2.2, label=str(dataset))

for ax, ylabel in zip(axes, ["Search time (ms)", "Projection time (ms)", "QP solved per query"]):
    ax.set_xscale("log")
    ax.set_xticks(K_ORDER)
    ax.set_xticklabels([str(k) for k in K_ORDER])
    ax.set_xlabel("k_per_class")
    ax.set_ylabel(ylabel)
    ax.legend(title="dataset")

fig.suptitle("Query decomposition: search is cheap, projection dominates", y=1.03, fontsize=14)
plt.tight_layout()
savefig("query_decomposition_vs_k.png")
plt.show()

## Candidate Pruning

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)

for dataset, ds_summary in time_summary.groupby("dataset", observed=True):
    k = ds_summary["k_per_class"].astype(int)
    axes[0].plot(k, ds_summary["candidates_total_mean"], marker="o", linewidth=2.2, label=f"{dataset}: total")
    axes[0].plot(k, ds_summary["candidates_considered_mean"], marker="s", linewidth=2.0, linestyle="--", label=f"{dataset}: considered")
    axes[1].plot(k, ds_summary["candidates_considered_mean"], marker="o", linewidth=2.2, label=str(dataset))

for ax in axes:
    ax.set_xscale("log")
    ax.set_xticks(K_ORDER)
    ax.set_xticklabels([str(k) for k in K_ORDER])
    ax.set_xlabel("k_per_class")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Candidate count")
axes[1].set_ylabel("Candidates considered / QPs solved")
axes[1].set_title("Effective solver load after pruning")

fig.suptitle("Candidate pruning as k grows", y=1.03, fontsize=14)
plt.tight_layout()
savefig("candidate_pruning_vs_k.png")
plt.show()

## Per-Query Winners

This plot answers a paired question: for the same dataset and the same query point, which `k_per_class` produced the smallest proximity?

For each query, we compare the counterfactuals generated by all tested `k_per_class` values. The `winner` is the `k` with the lowest distance for that query. We do this separately for `l1_distance` and `l2_distance`, so the L1 winner and L2 winner can differ.

How to read it:

- A bar counts how many queries were won by that `k_per_class`.
- If high `k` wins often, then increasing atlas support is not only improving the mean: it is helping many individual queries.
- If low `k` still wins often, then random sampling variance matters and larger atlases are not uniformly better.
- This is not a runtime metric. It is a per-query proximity comparison.

In [ ]:
winner_rows = []
for metric in ["l1_distance", "l2_distance"]:
    for dataset, ds in success_df.groupby("dataset", observed=True):
        winner_idx = ds.groupby("query_idx")[metric].idxmin()
        winners = ds.loc[winner_idx, "k_per_class"].astype(int).value_counts().reindex(K_ORDER, fill_value=0)
        for k, count in winners.items():
            winner_rows.append({"metric": metric, "dataset": dataset, "k_per_class": k, "wins": count})

winner_df = pd.DataFrame(winner_rows)
display(winner_df.pivot_table(index=["dataset", "metric"], columns="k_per_class", values="wins", fill_value=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, metric in zip(axes, ["l1_distance", "l2_distance"]):
    sub = winner_df.loc[winner_df["metric"] == metric]
    x = np.arange(len(K_ORDER))
    width = 0.24
    for offset, dataset in zip([-width, 0, width], DATASET_ORDER):
        values = sub.loc[sub["dataset"] == dataset].set_index("k_per_class").reindex(K_ORDER)["wins"]
        ax.bar(x + offset, values, width=width, label=dataset)
    ax.set_xticks(x)
    ax.set_xticklabels([str(k) for k in K_ORDER])
    ax.set_xlabel("k_per_class")
    ax.set_ylabel("number of winning queries")
    ax.set_title(metric.replace("_", " "))
    ax.legend(title="dataset")

fig.suptitle("Which k wins per query?", y=1.03, fontsize=14)
plt.tight_layout()
savefig("per_query_winners.png")
plt.show()

## Optional Rough Comparison Against Previous k-Medoids Run

This is not necessarily query-paired if the old benchmark used a different number of queries. Treat it as a rough orientation, not a definitive paired statistical comparison.

In [ ]:
if OLD_KMEDOIDS_PATH.exists():
    old_df = pd.read_parquet(OLD_KMEDOIDS_PATH)
    old_input = old_df.loc[
        old_df["success"] & old_df["method"].astype(str).str.contains("certcf_input_kmedoids", na=False)
    ].copy()

    old_summary = (
        old_input.groupby("dataset")
        .agg(
            old_l1_mean=("l1_distance", "mean"),
            old_l2_mean=("l2_distance", "mean"),
            old_runtime_mean_s=("runtime_s", "mean"),
        )
        .reset_index()
    )

    best_random = (
        proximity_summary.sort_values(["dataset", "l1_mean"])
        .groupby("dataset", observed=True)
        .head(1)[["dataset", "k_per_class", "l1_mean", "l2_mean"]]
        .rename(columns={"k_per_class": "best_random_k", "l1_mean": "random_l1_mean", "l2_mean": "random_l2_mean"})
    )

    comparison = best_random.merge(old_summary, on="dataset", how="left")
    comparison["l1_delta_random_minus_old"] = comparison["random_l1_mean"] - comparison["old_l1_mean"]
    comparison["l2_delta_random_minus_old"] = comparison["random_l2_mean"] - comparison["old_l2_mean"]
    display(comparison.style.format(precision=4))
else:
    print(f"Old k-medoids parquet not found: {OLD_KMEDOIDS_PATH}")

## Takeaways To Check

- Does proximity improve monotonically with `k_per_class`?
- Does the full distribution move, or only the mean?
- Does query time scale with all candidates, or only with post-pruning QPs?
- At what `k_per_class` does each dataset saturate?